In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
from fractions import Fraction
import ipywidgets as widgets
import math

# 1. Setup the Interactive UI Layout Elements
instruction_label = widgets.HTML(
    value="<div style='font-size:12px; color:gray; margin-bottom: 5px;'>"
          "<b>Examples you can try:</b> Whole Numbers (<code>4</code>), "
          "Decimals (<code>2.5</code>), Proper Fractions (<code>2/3</code>), "
          "Improper Fractions (<code>7/3</code>), or Irrationals (<code>pi</code>, <code>e</code>, <code>sqrt2</code>, <code>sqrt5</code>)"
          "</div>"
)

# Prompts the students to keep exploring continuously
next_step_label = widgets.HTML(
    value="<div style='font-size:13px; font-weight:bold; color:darkblue; margin-bottom: 5px;'>"
          "💬 Ready for another round! Enter your next number below:</div>"
)

input_box = widgets.Text(
    value='2/3',
    placeholder='e.g., 7/3, pi, sqrt2',
    description='Enter Number:',
    disabled=False
)

submit_button = widgets.Button(
    description='Locate Number',
    disabled=False,
    button_style='success', 
    icon='search'
)

# Stack the controls so the workflow feels natural to kids
ui_layout = widgets.VBox([instruction_label, next_step_label, widgets.HBox([input_box, submit_button])])
output_area = widgets.Output()

# 2. Main Logic Engine
def process_and_animate(b):
    with output_area:
        student_input = input_box.value.strip()
        clean_input = student_input.lower()
        
        # Clear out previous frames or errors cleanly
        clear_output(wait=True)
        
        # FIX: Immediately reset the text box so students are prompted for their next entry right away
        input_box.value = ''
        
        line_start = 0
        line_end = 5
        is_irrational = False
        irrational_name = ""

        # Catch irrational options
        if clean_input in ['pi', 'π']:
            is_irrational = True
            irrational_name = "π (Pi ≈ 3.14159...)"
        elif clean_input == 'e':
            is_irrational = True
            irrational_name = "e (Euler's Number ≈ 2.71828...)"
        elif clean_input in ['sqrt2', 'sqrt(2)']:
            is_irrational = True
            irrational_name = "√2 (Square Root of 2 ≈ 1.41421...)"
        elif clean_input in ['sqrt5', 'sqrt(5)']:
            is_irrational = True
            irrational_name = "√5 (Square Root of 5 ≈ 2.23606...)"
        elif clean_input in ['sqrt7', 'sqrt(7)']:
            is_irrational = True
            irrational_name = "√7 (Square Root of 7 ≈ 2.64575...)"

        # --- CONDITION A: IRRATIONAL ERROR DIALOGUE ---
        if is_irrational:
            fig, ax = plt.subplots(figsize=(10, 4.5))
            ax.set_xlim(-1, 1)
            ax.set_ylim(-1, 1)
            ax.axis('off')
            
            explanation = (
                f"🚨 MATHEMATICAL REALIZATION! 🚨\n\n"
                f"You chose the IRRATIONAL number: {irrational_name}\n\n"
                f"They do not belong on the number line of integers and rational numbers.\n"
                f"Because their decimal expansions go on forever without repeating,\n"
                f"they can never be written as a clean fraction of two whole numbers.\n\n"
                f"Furthermore, there is an infinite amount of irrational numbers and we\n"
                f"cannot list all of them! They are UNCOUNTABLY INFINITE, which is a\n"
                f"much larger size of infinity than the countably infinite integers and rationals."
            )
            ax.text(0, 0, explanation, ha='center', va='center', fontsize=11, fontweight='bold',
                    color='darkred', bbox=dict(facecolor='#ffe6e6', edgecolor='red', boxstyle='round,pad=1.5'))
            plt.close()
            display(fig)

        # --- CONDITION B: RATIONAL NUMBER TIMELINE ---
        else:
            try:
                frac_obj = Fraction(student_input)
                parsed_num = float(frac_obj)
                denominator = frac_obj.denominator
            except Exception:
                try:
                    parsed_num = float(student_input)
                    frac_obj = Fraction(student_input).limit_denominator(100)
                    denominator = frac_obj.denominator
                except ValueError:
                    print("❌ Formatting Error: Type a valid entry like '4', '2.5', '2/3', 'pi', or 'sqrt2'.")
                    return

            if parsed_num < line_start or parsed_num > line_end:
                print(f"⚠️ Out of Bounds! Please select a value between {line_start} and {line_end}.")
                return

            lower_int = math.floor(parsed_num)
            upper_int = lower_int + 1

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5))
            fig.suptitle(f"Locating '{student_input}' on the Number Line", fontsize=14, fontweight='bold')

            # Top View: Axis
            ax1.set_xlim(line_start - 0.5, line_end + 0.5)
            ax1.set_ylim(-0.5, 0.5)
            ax1.axhline(0, color='black', linewidth=2)
            ax1.axis('off')
            
            for tick in range(line_start, line_end + 1):
                ax1.plot([tick, tick], [-0.1, 0.1], color='black', linewidth=1.5)
                ax1.text(tick, -0.15, str(tick), ha='center', va='top', fontsize=11, fontweight='bold')
            ax1.axvspan(lower_int, upper_int, color='lightblue', alpha=0.3)

            # Bottom View: Magnified Sub-grid
            ax2.set_xlim(lower_int - 0.05, upper_int + 0.05)
            ax2.set_ylim(-0.5, 0.5)
            ax2.axhline(0, color='darkblue', linewidth=2)
            ax2.axis('off')

            ax2.plot([lower_int, lower_int], [-0.15, 0.15], color='darkblue', linewidth=2.5)
            ax2.text(lower_int, -0.2, str(lower_int), ha='center', va='top', fontsize=12, fontweight='bold', color='darkblue')
            ax2.plot([upper_int, upper_int], [-0.15, 0.15], color='darkblue', linewidth=2.5)
            ax2.text(upper_int, -0.2, str(upper_int), ha='center', va='top', fontsize=12, fontweight='bold', color='darkblue')

            sub_ticks = np.linspace(lower_int, upper_int, denominator + 1)
            for idx, sub_x in enumerate(sub_ticks[1:-1]):
                ax2.plot([sub_x, sub_x], [-0.08, 0.08], color='purple', linestyle='-', linewidth=1.2)
                ax2.text(sub_x, -0.18, f"+{idx+1}/{denominator}", ha='center', va='top', fontsize=9, color='purple')

            main_point, = ax1.plot([], [], 'ro', markersize=12)
            zoom_point, = ax2.plot([], [], 'ro', markersize=14)
            zoom_line, = ax2.plot([], [], color='red', linestyle='--', linewidth=1.5)

            status_box = ax2.text((lower_int + upper_int)/2, 0.35, '', ha='center', va='center', 
                                  fontsize=11, fontweight='bold', color='darkblue',
                                  bbox=dict(facecolor='lightyellow', alpha=0.8, boxstyle='round,pad=0.4'))

            num_frames = 40
            path_main = np.linspace(line_start, parsed_num, num_frames)
            path_zoom = np.linspace(lower_int, parsed_num, num_frames)

            def update_dual_line(frame):
                curr_m = path_main[frame]
                curr_z = path_zoom[frame]
                main_point.set_data([curr_m], [0.0])
                zoom_point.set_data([curr_z], [0.0])
                zoom_line.set_data([curr_z, curr_z], [0.0, 0.25])
                status_box.set_text(f"Scanning... Tracking Coordinate: {curr_m:.3f}")
                
                if frame == num_frames - 1:
                    status_box.set_text(f"🎯 Found! '{student_input}' sits precisely at coordinate {parsed_num:.4f}")
                    status_box.set_backgroundcolor('#e6ffe6')
                    
                return main_point, zoom_point, zoom_line, status_box

            ani_dual = FuncAnimation(fig, update_dual_line, frames=num_frames, interval=50, blit=True, repeat=False)
            plt.close()
            display(HTML(ani_dual.to_jshtml()))

# Attach the click callback handler
submit_button.on_click(process_and_animate)

# Final block paint
display(ui_layout, output_area)


Output()